### Overview
We will perform supervised land cover classification of Sentinel-2 imagery using a Random Forest classifier. A pre-made cloud-free median composite is loaded directly as a multi-band GeoTIFF. Labeled training points (Ground Control Points) provide one spectral sample per class location. We extract band values at those points, train the classifier, and apply it across the full image to produce a land cover map.

### Setup and Data Download

The following blocks of code will install the required packages and download the datasets to your Colab environment.

In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip install rioxarray scikit-learn

In [ ]:
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import rioxarray as rxr
import xarray as xr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
data_folder = 'data'
output_folder = 'output'

if not os.path.exists(data_folder):
    os.mkdir(data_folder)
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

### Load the Composite

We load a pre-made cloud-free Sentinel-2 median composite for 2019 using [`rioxarray`](https://corteva.github.io/rioxarray/stable/). The file is a Cloud Optimized GeoTIFF (COG), so only the metadata is fetched until pixels are actually needed.

In [ ]:
composite_url = (
    'https://storage.googleapis.com/spatialthoughts-public-data/'
    'python-remote-sensing/composite_2019.tif'
)
composite = rxr.open_rasterio(composite_url, mask_and_scale=True)
composite

The composite has integer band indices (1, 2, 3 …). We assign meaningful names so we can select bands by name later.

In [ ]:
band_names = ['blue', 'green', 'red', 'nir', 'swir16', 'swir22']
composite = composite.assign_coords(band=band_names)
composite

### Load Training Data

The training data is a set of Ground Control Points (GCPs) — point features, each labeled with a land cover class. We load the GeoJSON file with GeoPandas.

In [ ]:
training_url = (
    'https://storage.googleapis.com/spatialthoughts-public-data/'
    'python-remote-sensing/bangalore_gcps.geojson'
)
training_gdf = gpd.read_file(training_url)
training_gdf.head()

In [ ]:
print('Columns:', training_gdf.columns.tolist())
print('CRS:', training_gdf.crs)
print('\nClass distribution:')
print(training_gdf['classname'].value_counts())

### Visualize Training Data on Composite

We reproject the training points to match the composite CRS, then overlay them on an RGB preview of the composite. This lets us verify that the training points cover the expected land cover types.

In [ ]:
image_crs = composite.rio.crs
training_gdf_proj = training_gdf.to_crs(image_crs)

In [ ]:
# Downsample for faster display
preview = composite.rio.reproject(composite.rio.crs, resolution=100)

class_colors = {
    'Urban': '#E0E0E0',
    'Vegetation': '#228B22',
    'Water': '#1E90FF',
    'Bare Land': '#D2B48C',
}

fig, ax = plt.subplots(1, 1)
fig.set_size_inches(7, 7)

preview.sel(band=['red', 'green', 'blue']).plot.imshow(
    ax=ax,
    robust=True,
)

for classname, group in training_gdf_proj.groupby('classname'):
    group.plot(
        ax=ax,
        color=class_colors.get(classname, 'red'),
        markersize=5,
        label=classname,
    )

ax.legend(loc='upper left')
ax.set_title('Training Points on Sentinel-2 RGB Composite (2019)')
ax.set_axis_off()
ax.set_aspect('equal')
plt.show()